In [3]:
from pathlib import Path
from models import CifarResNET18
import torch
import numpy as np
import random
from metrics import calibration_errors, nll_loss
from dataset_utils import get_data_loaders

In [9]:
train_loader, test_loader = get_data_loaders(dataset="cifar100")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
directory_path = Path("resnet") 

seed = 0
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 20 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [18]:
for filepath in directory_path.iterdir():
    if filepath.is_file():  # Check if the entry is a file
        model = CifarResNET18(num_classes=100).to(device)
        model.load_state_dict(torch.load(filepath, weights_only=True))
        model.eval()

        print(filepath)
        logits_list = []
        labels_list = []

        model.eval()
        with torch.no_grad():
            for x, y in test_loader:
                x = x.to(device)
                y = y.to(device)

                logits = model(x)

                logits_list.append(logits)
                labels_list.append(y)

        logits_all = torch.cat(logits_list, dim=0)
        labels_all = torch.cat(labels_list, dim=0)

        ces = calibration_errors(logits_all, labels_all, n_bins=15)
        print("ECE:", 100 * ces[0])
        print("MCE:", 100 * ces[1])
        print("NLL:", nll_loss(logits_all, labels_all))
        print()


resnet/vae4_002_1_0.pth
ECE: 2.472541853785515
MCE: 12.888522446155548
NLL: 0.8659029006958008

resnet/vae4_005_5_0.pth
ECE: 6.097767502069473
MCE: 15.31417965888977
NLL: 0.8927049040794373

resnet/vae4_002_10_0.pth
ECE: 4.463499411940575
MCE: 15.529334545135498
NLL: 0.8757277131080627

resnet/vae4_001_3_0.pth
ECE: 3.31430584192276
MCE: 14.984217286109924
NLL: 0.8546327948570251

resnet/ls001_0.pth
ECE: 4.455408826470375
MCE: 18.278001248836517
NLL: 0.893711507320404

resnet/ls_01_0.pth
ECE: 13.096646964550018
MCE: 27.454790472984314
NLL: 1.0456011295318604

resnet/ce_0.pth
ECE: 4.751721769571304
MCE: 12.923778593540192
NLL: 0.8776656985282898

resnet/ls_0005_0.pth
ECE: 4.13145087659359
MCE: 18.643248081207275
NLL: 0.894155740737915

resnet/ls005_0.pth
ECE: 8.972946554422379
MCE: 22.788920998573303
NLL: 0.9566543102264404

resnet/vae4_001_1_0.pth
ECE: 3.4765150398015976
MCE: 15.331023931503296
NLL: 0.8631239533424377

resnet/vae4_001_10_0.pth
ECE: 3.970053419470787
MCE: 17.422433197498

In [8]:
!ls

data  dataset_utils.py	metrics.py  models.py  __pycache__  resnet  sample_data
